# Prepare the public deck comparison

Compare your deck to individual published builds. The default user deck is Who Killed Kenny?, the creator's five-color Kenrith list. It is never part of the reference sample. This notebook writes the shared data for three separately built dashboards.

## Your deck: change this one path

Kenrith is the example player input, not a reference. To analyze your own 100, change only `user_deck_path` below, then run all cells. Use the same Commander and Deck headings as the example, one quantity and English card name per line. Leave the 19 saved reference lists as they are. Cached cards run offline; new names require Scryfall access.

In [ ]:
import csv
from collections import Counter
from pathlib import Path

# PLAYER INPUT: leave Kenrith here to reproduce the example.
user_deck_path = Path("data/comparison/Who Killed Kenny - Creator's Base Deck.txt")
reference_catalog = Path("data/reference_decks.csv")
results_path = Path("results/deck_comparisons.csv")

## Compare card choices

Each shared name counts once, including commanders and basics. Match full names case-insensitively; use the same spelling and double-faced naming format in both lists. This first measure describes shared card choices, not deck power. Keep the precon rows labeled separately; a blank bracket means none was supplied by the source.

In [ ]:
def read_deck(deck_path):
    card_counts = Counter()
    for line_number, line in enumerate(deck_path.read_text(encoding="utf-8-sig").splitlines(), start=1):
        entry = line.strip()
        if not entry or entry.casefold() in {"commander", "deck"}:
            continue
        fields = entry.split(maxsplit=1)
        if len(fields) != 2 or not fields[0].isdecimal() or int(fields[0]) < 1:
            raise ValueError(f"{deck_path.name}, line {line_number}: expected a positive quantity and card name")
        quantity, card_name = fields
        card_counts[card_name.casefold()] += int(quantity)
    return card_counts

In [ ]:
user_cards = read_deck(user_deck_path)
if sum(user_cards.values()) != 100:
    raise ValueError(f"{user_deck_path.name}: expected 100 cards including the commander")

with reference_catalog.open(encoding="utf-8-sig", newline="") as catalog_file:
    reference_rows = list(csv.DictReader(catalog_file))

user_label = user_deck_path.stem
if user_label in {r["reference_deck"] for r in reference_rows}:
    user_label += " (your deck)"
comparison_rows = []
reference_names = set()
for reference in reference_rows:
    deck_name = reference["reference_deck"]
    group = reference["reference_group"]
    bracket = reference["source_bracket"]
    if deck_name in reference_names or not deck_name:
        raise ValueError(f"Duplicate or missing reference name: {deck_name!r}")
    if group not in {"Player-built", "Precon"} or bracket not in {"2", "3", "4", ""}:
        raise ValueError(f"{deck_name}: check the group or bracket")
    if group == "Player-built" and not bracket:
        raise ValueError(f"{deck_name}: include the author's stated bracket (2, 3 or 4)")
    reference_cards = read_deck(Path(reference["deck_file"]))
    if sum(reference_cards.values()) != 100:
        raise ValueError(f"{deck_name}: expected 100 cards including the commander")
    shared_names = user_cards.keys() & reference_cards.keys()
    cohort_label = f"Bracket {bracket}" if group == "Player-built" else "Precon"
    comparison_rows.append({
        "user_deck": user_label,
        "reference_deck": deck_name,
        "reference_label": f"{cohort_label}: {deck_name}",
        "reference_group": group,
        "source_bracket": bracket,
        "shared_card_names": len(shared_names),
        "source_url": reference["source_url"],
    })
    reference_names.add(deck_name)

if not comparison_rows:
    raise ValueError("The reference catalog has no decks.")
results_path.parent.mkdir(parents=True, exist_ok=True)
with results_path.open("w", encoding="utf-8", newline="") as results_file:
    writer = csv.DictWriter(results_file, fieldnames=list(comparison_rows[0]))
    writer.writeheader()
    writer.writerows(comparison_rows)

print(f"Wrote {len(comparison_rows)} comparisons to {results_path}.")

## Gameplay profiles

Use `Commander` and `Deck` headings in the input lists. The shipped sample runs offline. New names are resolved from Scryfall in batches; set `refresh_metadata` to refresh the current cards and Game Changer flags. Counts are cards, not plays.

In [ ]:
import hashlib
import json
import re
import time
import unicodedata
from datetime import date
from statistics import mean

refresh_metadata = False
cache_path = Path("data/card_facts.json")
deck_inputs = [(user_label, "User", "", user_deck_path)] + [
    (row["reference_deck"], row["reference_group"], row["source_bracket"], Path(row["deck_file"]))
    for row in reference_rows
]

def normalize(name):
    result = unicodedata.normalize("NFC", name.strip()).replace("\u2019", "'").casefold()
    return result

def read_sections(path):
    sections = {"Commander": Counter(), "Deck": Counter()}
    section = None
    for line in path.read_text(encoding="utf-8-sig").splitlines():
        line = line.strip()
        if not line:
            continue
        if line.casefold() in {"commander", "deck"}:
            section = line.capitalize()
        else:
            if section is None:
                raise ValueError(f"{path}: add Commander and Deck headings for the gameplay profile")
            quantity, name = line.split(maxsplit=1)
            sections[section][name.strip()] += int(quantity)
    if not 1 <= sum(sections["Commander"].values()) <= 2 or not sections["Deck"]:
        raise ValueError(f"{path}: check commander and library sections")
    return sections

deck_sections = [read_sections(path) for _, _, _, path in deck_inputs]
requested_names = sorted({n for deck in deck_sections for section in deck.values() for n in section})
commander_names = {normalize(n) for deck in deck_sections for n in deck["Commander"]}


In [ ]:
def get_card_facts(names, commander_names, cache_path, refresh=False):
    facts = json.loads(cache_path.read_text(encoding="utf-8")) if cache_path.exists() else {}
    missing = [n for n in names if refresh or normalize(n) not in facts
               or "printed_colors" not in facts[normalize(n)]
               ]
    if missing:
        import requests
        headers = {"User-Agent": "CommanderPortfolioAnalysis/1.0", "Accept": "application/json"}
        for offset in range(0, len(missing), 70):
            batch = missing[offset:offset+70]
            response = requests.post("https://api.scryfall.com/cards/collection",
                                     json={"identifiers": [{"name": n.split(" // ")[0]} for n in batch]},
                                     headers=headers, timeout=30)
            response.raise_for_status()
            catalog = response.json()
            aliases = {}
            for card in catalog["data"]:
                faces = card.get("card_faces", [])
                for alias in [card["name"], *(f["name"] for f in faces)]:
                    aliases[normalize(alias)] = card
            for name in batch:
                key = normalize(name)
                if key not in aliases:
                    raise ValueError(f"Scryfall did not resolve {name!r}; check the exact English name")
                card = aliases[key]
                faces = card.get("card_faces", [])
                front = faces[0] if faces and card["layout"] != "split" else card
                rules = card.get("oracle_text") or "\n".join(f.get("oracle_text","") for f in faces)
                facts[key] = {"name": card["name"], "oracle_id": card["oracle_id"],
                              "faces": [{"name":f["name"],"cost":f.get("mana_cost",""),"type":f["type_line"]} for f in (faces or [card])],
                              "printed_colors": front["colors"] if "colors" in front else card["colors"], "identity": card["color_identity"],
                              "printed_cost": front.get("mana_cost") or "", "layout":card["layout"],
                              "url": card["scryfall_uri"].split("?")[0], "oracle_text": rules, "keywords": card.get("keywords", []),
                              "all_types": card["type_line"],
                              "type": front["type_line"], "mv": card["cmc"],
                              "land_back": card["layout"] == "modal_dfc" and "Land" in faces[-1]["type_line"].split(" — ")[0].split(),
                              "game_changer": card["game_changer"],
                              "legal": card["legalities"]["commander"], "as_of": date.today().isoformat()}
                if key in commander_names:
                    facts[key]["commander_text"] = rules
            time.sleep(0.12)
        cache_path.write_text(json.dumps(facts, ensure_ascii=False, sort_keys=True, indent=1)+"\n", encoding="utf-8")
    return facts

card_facts = get_card_facts(requested_names, commander_names, cache_path, refresh_metadata)


In [ ]:
role_axes = {
    "Deployment routes": ("deployment", ["Instant / flash spell","Own alternate payment","Deploy other cards","Spell discounts","Ability discounts","Hand ability"]),
    "Interaction targets": ("targets", ["Creatures","Artifacts","Enchantments","Planeswalkers","Lands","Graveyards","Hands","Stack spells","Stack abilities"]),
    "Interaction methods": ("methods", ["Destroy","Exile / sacrifice","Bounce / tuck","Damage / shrink","Tap / disable","Temporary exile / phase","Steal / transform","Counter / stack","Multi-permanent reset","Taxes / restrictions","Protect board","Graveyard disruption","Enable targeting / damage","Combat prevention / redirection","Hand disruption"]),
    "Recovery routes": ("recovery", ["Recover other cards","Own-card reuse","Blink / return","Copy / imprint","Untap support","Recurring draw","Entry draw","Token production","Token modifiers","Return own to hand"])
}
reviewed_roles = json.loads(Path("data/reviewed_card_roles.json").read_text(encoding="utf-8"))
def printed_roles(card):
    # Classify distinct capabilities, not strength or expected uses. Explicit objects avoid matching costs as answers.
    result = {key:set() for key,_ in role_axes.values()}
    if card["oracle_id"] in reviewed_roles:
        for key,tags in reviewed_roles[card["oracle_id"]].items():
            result[key].update(tags)
        return result, "Card annotation"
    text = re.sub(r"\([^()]*\)", "", card["oracle_text"].lower())
    keys = {k.lower() for k in card["keywords"]}
    face = card["all_types"].lower()
    has = lambda pattern: bool(re.search(pattern,text))
    d,t,m,r = (result[key] for key in ("deployment","targets","methods","recovery"))
    if "instant" in face or "flash" in keys or has(r"cast this spell as though it had flash"): d.add("Instant / flash spell")
    if keys & {"affinity","convoke","foretell","delve","improvise","suspend","evoke","plot","miracle","prototype","emerge","morph","dash","spectacle"} or has(r"cast this spell without paying its mana cost|rather than pay (?:this spell.s|its) mana cost|this spell costs [^\n]*?less to cast"): d.add("Own alternate payment")
    if has(r"(?:put|return) (?!this card)[^\n.]{0,110}(?:from (?:your hand|a graveyard|your graveyard)|cards? revealed this way)[^\n.]{0,45}(?:onto|to) the battlefield|cast (?:it|that card|a copy|a spell|the copy)[^\n.]{0,60}without paying") or keys & {"discover","cascade","reanimate"}: d.add("Deploy other cards")
    if has(r"spells [^\n.]{0,100}cost [^\n.]{0,40}less to cast|spells [^\n.]*have (?:affinity|convoke|improvise)"): d.add("Spell discounts")
    if has(r"(?:abilities|equip costs) [^\n.]{0,90}cost [^\n.]*less|equip abilities [^\n.]*cost"): d.add("Ability discounts")
    if any("cycling" in k or k in {"ninjutsu","channel","transmute","reinforce","bloodrush"} for k in keys) or has(r"channel\s*[—-]"): d.add("Hand ability")
    def objects(clause):
        targets = set()
        for key,noun in zip(["Creatures","Artifacts","Enchantments","Planeswalkers","Lands"],
                            ["creature","artifact","enchantment","planeswalker","land"]):
            if re.search(r"\b"+noun+r"s?\b",clause): targets.add(key)
        if re.search(r"\bpermanents?\b",clause):
            targets.update(["Creatures","Artifacts","Enchantments","Planeswalkers"])
            if "nonland" not in clause: targets.add("Lands")
        return targets
    for line in re.split(r"[.\n;]",text):
        # Restrict to the effect, not the sacrifice/tap before an activated ability's colon.
        effect = line.split(":")[-1]
        for action,method in [(r"destroy","Destroy"),(r"exile","Exile / sacrifice"),(r"return","Bounce / tuck"),
                              (r"tap","Tap / disable"),(r"gain control of","Steal / transform")]:
            for match in re.finditer(r"\b"+action+r" ((?:up to |any number of |any |each |all |another |other |target |a |two |three |x |an?)(?:[^.;\n]{0,145}))",effect):
                clause = re.split(r",| then | and (?:you|draw|create|exile|return|destroy)| to (?:its|their|your|the)| from | with ",match[1])[0]
                if re.search(r"\byou (?:own|control)\b|\byour (?:hand|graveyard|library)\b",clause): continue
                if action in ("exile","return") and re.search(r"from (?:a|your|that|any|each|an opponent.s) graveyard|in a graveyard|cards? in [^.;]*graveyard",effect): continue
                if action=="return" and not re.search(r"hand|librar",effect): continue
                hits=objects(clause)
                if hits:
                    t.update(hits);m.add(method)
                    if action in ("destroy","exile","return") and re.search(r"^(all|each|every|any number|up to [2-9]|two |x )",clause): m.add("Multi-permanent reset")
        if re.search(r"(?:target (?:opponent|player)|each (?:opponent|player)|that player) sacrifices? (?:a|an|all|half|two)",effect):
            hits=objects(effect.split("sacrifice")[-1]);t.update(hits)
            if hits:m.add("Exile / sacrifice")
        if re.search(r"(?:deals?|deal) [^;.]{0,75}damage to (?:any target|(?:each|target|all|up to) [^;.]{0,55}(?:creature|planeswalker))|(?:target|all|each) creatures?[^;.]{0,55}gets? -|fights? (?:another |up to |target )",effect):
            t.add("Creatures");m.add("Damage / shrink")
            if "any target" in effect or "planeswalker" in effect:t.add("Planeswalkers")
            if re.search(r"damage to (?:each|all) creature|all creatures get -",effect):m.add("Multi-permanent reset")
        if re.search(r"(?:counter|exile|return) (?:any number of |up to [^\s]+ |each |all |target )[^.;]{0,35}\b(?:spell|activated ability|triggered ability)",effect):
            m.add("Counter / stack")
            if "spell" in effect:t.add("Stack spells")
            if "ability" in effect:t.add("Stack abilities")
        if re.search(r"(?:opponent|each player|target player)[^.;]{0,30}discards?",effect):
            m.add("Hand disruption");t.add("Hands")
        if re.search(r"exile (?:all |target |up to |each |a |any number)[^.;]{0,70}(?:graveyard)|cards? in graveyards? can.t",effect) and "your graveyard" not in effect:
            m.add("Graveyard disruption");t.add("Graveyards")
        if re.search(r"(?:opponents?|players?|creatures?) [^.;]{0,40}can.t (?:cast|activate|attack|block|draw|untap)|spells your opponents cast cost",effect): m.add("Taxes / restrictions")
        if re.search(r"(?:target|creatures you control|permanents you control)[^.;]{0,70}(?:gains?|have) (?:hexproof|indestructible|protection)",effect): m.add("Protect board")
        if re.search(r"phase out|phases out",effect): m.add("Temporary exile / phase")
        if re.search(r"prevent all combat damage|reselect which player",effect):m.add("Combat prevention / redirection")
        if re.search(r"lose (?:hexproof|indestructible|shroud)",effect):m.add("Enable targeting / damage")
    if has(r"(?:return|cast|play|put) [^.\n]{0,150}(?:from (?:a|any|your|an opponent.s) graveyard|cards? in your graveyard)") and not has(r"(?:cast|return|play) this card from your graveyard") or keys & {"reanimate","unearth"} and has("target creature card"):r.add("Recover other cards")
    if keys & {"flashback","escape","retrace","unearth","disturb","embalm","eternalize","persist","undying","encore","jump-start","aftermath"} or has(r"(?:cast|return|play) this card from your graveyard"):r.add("Own-card reuse")
    if has(r"exile [^.\n]{0,100}(?:creature|permanent)[\s\S]{1,125}return (?:it|that card|those cards|them) to the battlefield"):r.add("Blink / return")
    if has(r"\bcopy (?:target|that|it|the|each|an? |any|this|the next)|(?:a|that's a) copy of") or "imprint" in keys:r.add("Copy / imprint")
    if has(r"\buntap (?:target|up to|all|another|each|it|that permanent)") :r.add("Untap support")
    if has(r"return (?:this (?:permanent|creature)|[^.\n]{0,60}you (?:control|own)) to (?:its owner.s|your) hand"):r.add("Return own to hand")
    if has(r"(?:when |whenever |at the beginning of |: )[^.\n]*\bdraw (?:a|two|three|cards|x)") and not has(r"^as an additional cost") and re.search("creature|artifact|enchantment|land|planeswalker",face):
        if has(r"when [^.\n]{0,80}enters[^.\n]*\bdraw"):r.add("Entry draw")
        if has(r"(?:whenever |at the beginning of |: )[^.\n]*\bdraw") and not has(r"sacrifice this [^:\n]*: draw"):r.add("Recurring draw")
    if has(r"if (?:one or more|an effect would create|you would create)[^.\n]*tokens?[^.\n]*instead"):r.add("Token modifiers")
    if (has(r"\bcreate [^.\n]{0,160}\btokens?") or keys & {"investigate","amass","incubate","fabricate","living weapon","populate"}) and not has(r"(?:its controller|that player|each opponent|target opponent) creates?|if (?:one or more|an effect would create)[^.\n]*tokens?[^.\n]*instead"):r.add("Token production")
    return result, "Oracle text rule"

def library_summary(audit):
    # Describe demonstrated library capabilities, not a guessed win condition.
    examples = {"Deploy other cards":[], "Recurring draw":[], "Token production":[], "Recover other cards":[], "Counter / stack":[]}
    for item in audit:
        if item["zone"] == "Deck":
            roles, source = printed_roles(card_facts[normalize(item["name"])])
            abilities = set().union(*roles.values())
            for ability in examples:
                if ability in abilities:
                    examples[ability].append(item["name"])
    phrases = {"Deploy other cards":"can put other cards into play outside their normal casting cost",
               "Recurring draw":"offer recurring draw","Token production":"can create tokens",
               "Recover other cards":"can recover other cards","Counter / stack":"can interact on the stack"}
    strongest = sorted((k for k in examples if examples[k]), key=lambda k:-len(examples[k]))[:2]
    result = "; ".join(f'{k}: {len(examples[k])} in the library (e.g. {", ".join(sorted(examples[k])[:2])})' for k in strongest)
    if not result:
        library_spells = [r for r in audit if r["curve_bin"]]
        early_slots = sum(r["quantity"] for r in library_spells if r["mv"]<=2)
        result = f"{early_slots} library spells cost two or less at their printed mana value. No independent engine was identified by the role tags."
    return result.rstrip(".")+"."


In [ ]:
curve_labels = ["0-1", "2", "3", "4", "5", "6+"]
type_labels = ["Land", "Creature", "Artifact", "Enchantment", "Instant", "Sorcery", "Planeswalker", "Battle"]
reviews = {r["signature"]: r for r in json.loads(Path("data/style_notes.json").read_text(encoding="utf-8"))}

def measure_deck(sections, use_reference_note=False):
    curve, types, tribes = Counter(), Counter(), Counter()
    audit, changers, illegal = [], [], []
    mv_total = land_backs = 0
    for zone, card_counts in sections.items():
        for name, quantity in card_counts.items():
            card = card_facts[normalize(name)]
            lines = [line.partition(" — ") for line in card["type"].split(" // ")]
            card_types = {t for face, _, _ in lines for t in face.split() if t in type_labels}
            subtypes = {t for face, _, subs in lines if "Creature" in face.split() for t in [v.replace("Time_Lord","Time Lord") for v in subs.replace("Time Lord","Time_Lord").split()]}
            mv = card["mv"]
            bin_name = ""
            if zone == "Deck" and "Land" not in card_types:
                bin_name = "0-1" if mv <= 1 else "6+" if mv >= 6 else str(int(mv))
                curve[bin_name] += quantity
                mv_total += quantity * mv
            for t in card_types:
                types[t] += quantity
            for t in subtypes:
                tribes[t] += quantity
            if card["game_changer"]:
                changers.append(name)
            if card["legal"] != "legal":
                illegal.append(name)
            extra_land_face = card["land_back"] and "Land" not in card_types
            land_backs += quantity if extra_land_face else 0
            audit.append(dict(zone=zone, quantity=quantity, name=name, oracle_id=card["oracle_id"],
                              mv=mv, curve_bin=bin_name, types="; ".join(sorted(card_types)),
                              subtypes="; ".join(sorted(subtypes)), optional_land_back=extra_land_face))
    library_spells = sum(curve.values())
    signature_text = sorted(f"{q} {n}" for n,q in sections["Commander"].items()) + ["Deck"] + sorted(
        f"{q} {n}" for n,q in sections["Deck"].items())
    signature = hashlib.sha256("\n".join(signature_text).encode()).hexdigest()
    note = reviews.get(signature, {}) if use_reference_note else {}
    commander_text = " / ".join(card_facts[normalize(n)]["oracle_text"] for n in sections["Commander"])
    if library_spells == 0:
        raise ValueError("A nonland library card is needed for a normalized curve comparison.")
    return dict(curve=curve, types=types, tribes=tribes, audit=audit, changers=sorted(changers),
                illegal=illegal, land_backs=land_backs, spells=library_spells, mean_mv=mv_total/library_spells,
                commander=" / ".join(sections["Commander"]),
                commander_mv=" / ".join(f'{card_facts[normalize(n)]["mv"]:g}' for n in sections["Commander"]),
                without=note["without"] if note else library_summary(audit),
                adds=note.get("adds", commander_text))

profiles = [dict(name=n, group=g, bracket=b, **measure_deck(sec, use_reference_note=g!="User"))
            for (n,g,b,_),sec in zip(deck_inputs,deck_sections)]


## Which table would I start with?

For each player-built bracket, take the per-deck mean for each curve bin (% of nonland library) and type (% of 100 slots). Distance is the mean absolute difference in percentage points within each family, averaged equally across the two families. Smaller means similar construction, not faster wins. Tribe names are descriptive, not a strength signal. Neither precons nor the user enter those means. Check Game Changers separately; this is not a legality, combo, or cEDH classifier. Discuss intended win timing and play patterns with the table.

In [ ]:
def construction_vector(profile):
    result = [100*profile["curve"][k]/profile["spells"] for k in curve_labels]
    result += [profile["types"][k] for k in type_labels]
    return result

def distance(a, b):
    gaps = [abs(x-y) for x,y in zip(a,b)]
    result = (mean(gaps[:len(curve_labels)]) + mean(gaps[len(curve_labels):])) / 2
    return result

user = profiles[0]
vector = construction_vector(user)
fit_rows = []
cohorts = {}
for bracket in ("4", "3", "2"):
    cohort = [p for p in profiles if p["group"] == "Player-built" and p["bracket"] == bracket]
    if not cohort:
        continue
    cohorts[bracket] = cohort
    center = [mean(col) for col in zip(*(construction_vector(p) for p in cohort))]
    neighbor = min(cohort, key=lambda p:distance(vector, construction_vector(p)))
    fit_rows.append(dict(bracket=bracket, decks=len(cohort), gap=distance(vector,center),
                         nearest=neighbor["name"], nearest_gap=distance(vector,construction_vector(neighbor))))
if not fit_rows:
    raise ValueError("Include at least one bracketed player-built reference.")
closest = min(fit_rows, key=lambda r:r["gap"])["bracket"]
changer_count = len(user["changers"])
eligible = [r for r in fit_rows if int(r["bracket"]) >= (4 if changer_count > 3 else 3 if changer_count else 2)]
suggestion = min(eligible, key=lambda r:r["gap"])["bracket"] if eligible else None
verdict = f"Potential table: bracket {suggestion}. Similar card costs and types do not establish an optimized game plan or a win turn." if suggestion else "No sampled bracket meets the Game Changer count."
if user["illegal"]:
    verdict = "Check Commander legality with the table before assigning a bracket: " + ", ".join(user["illegal"])


In [ ]:
def write_csv(path, rows):
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def leading_tribes(counts):
    ranking = sorted(counts.items(), key=lambda kv:(-kv[1],kv[0]))
    cutoff = ranking[min(2,len(ranking)-1)][1] if ranking else 0
    result = "; ".join(f"{n} ({v:g})" for n,v in ranking if v >= cutoff and v > 0)
    return result or "None"

groups = [("Your deck" if p["group"] == "User" else
           f"Bracket {p['bracket']}" if p["group"]=="Player-built" else "Stock precon") for p in profiles]
profile_rows, audit_rows, grid = [], [], []
def add_cell(view, deck, group, metric, display, number="", detail=""):
    grid.append(dict(view=view, deck=deck, group=group, metric=metric,
                     display=str(display), value=number, detail=detail))

def grid_profile(name, group, curve, types, tribes, mv, land_backs, spells):
    for label in curve_labels:
        val = curve.get(label,0)
        add_cell("Mana curve",name,group,label,f"{val:g}",val)
    add_cell("Mana curve",name,group,"Mean MV",f"{mv:.2f}",round(mv,2))
    for label in type_labels:
        val = types.get(label,0)
        add_cell("Card types",name,group,label,f"{val:g}",val)
    add_cell("Card types",name,group,"Land backs",f"{land_backs:g}",land_backs)
    add_cell("Tribes",name,group,"Leading printed subtypes",leading_tribes(tribes))
    
for p, group in zip(profiles,groups):
    n = p["name"]
    grid_profile(n,group,p["curve"],p["types"],p["tribes"],p["mean_mv"],p["land_backs"],p["spells"])
    add_cell("Commander roles",n,group,"Without casting the commander",p["without"])
    add_cell("Commander roles",n,group,"What the commander adds",p["adds"])
    add_cell("Commander roles",n,group,"Commander MV",p["commander_mv"])
    profile_rows.append(dict(deck=n,group=group,commander=p["commander"], commander_mv=p["commander_mv"],
                    library_nonlands=p["spells"],mean_mv=round(p["mean_mv"],3),
                    **{"MV "+k:p["curve"][k] for k in curve_labels},**{k:p["types"][k] for k in type_labels},
                    optional_land_backs=p["land_backs"],leading_tribes=leading_tribes(p["tribes"]),
                    without_commander=p["without"],commander_adds=p["adds"],game_changers="; ".join(p["changers"])))
    audit_rows.extend(dict(deck=n,**card) for card in p["audit"])
for b, cohort in cohorts.items():
    def avg(field, keys):
        return {k: mean(p[field][k] for p in cohort) for k in keys}
    grid_profile(f"Bracket {b} mean","Reference means",avg("curve",curve_labels),avg("types",type_labels),
                 avg("tribes",set().union(*(p["tribes"].keys() for p in cohort))),
                 mean(p["mean_mv"] for p in cohort),mean(p["land_backs"] for p in cohort),
                 mean(p["spells"] for p in cohort))
for r in fit_rows:
    b = f"Bracket {r['bracket']}"
    add_cell("Table fit",b,"Reference means","Lists",r["decks"],r["decks"])
    add_cell("Table fit",b,"Reference means","Gap (pp)",f"{r['gap']:.2f}",round(r["gap"],2))
    add_cell("Table fit",b,"Reference means","Closest list",r["nearest"])
    add_cell("Table fit",b,"Reference means","List gap (pp)",f"{r['nearest_gap']:.2f}",round(r["nearest_gap"],2))
match_text = (f"Closest construction: bracket {closest}. Game Changers ({changer_count}): "
              + (", ".join(user["changers"]) or "none") + ". " + verdict)
add_cell("Seating note",user["name"],"Your deck","Starting table",match_text)
out_dir = results_path.parent
write_csv(out_dir/"gameplay_profiles.csv",profile_rows)
write_csv(out_dir/"gameplay_cards.csv",audit_rows)
write_csv(out_dir/"gameplay_dashboard.csv",grid)
print(f"{len(profiles)} deck profiles; {len(audit_rows)} card rows. {match_text}")

In [ ]:
# Chart facts: each focus deck and the same reference means used in the detail tables.
chart_rows = []
def add_mark(plot, focus, name, category, series, quantity, position):
    chart_rows.append(dict(plot=plot, focus=focus, deck=name, category=category, series=series,
                           quantity=round(quantity,4), position=position))

for p in profiles:
    name = p["name"]
    focus = "Your deck" if p["group"] == "User" else name
    for plot, family, categories in [("Mana curve","curve",curve_labels), ("Card types","types",type_labels)]:
        for pos, category in enumerate(categories):
            add_mark(plot,focus,name,category,"Selected deck",p[family][category],pos)
            for b, cohort in cohorts.items():
                amount = mean(ref[family][category] for ref in cohort)
                add_mark(plot,focus,name,category,f"Bracket {b} mean",amount,pos)
    ranking = sorted(p["tribes"].items(),key=lambda kv:(-kv[1],kv[0]))
    cutoff = ranking[min(2,len(ranking)-1)][1] if ranking else 0
    for pos,(category,amount) in enumerate(ranking):
        if amount >= cutoff:
            add_mark("Leading tribes",focus,name,category,"Selected deck",amount,pos)
for pos,r in enumerate(fit_rows):
    add_mark("Table fit","Your deck",user["name"],"Bracket "+r["bracket"],"Gap (pp)",r["gap"],pos)
write_csv(out_dir/"gameplay_charts.csv",chart_rows)


In [ ]:
# The question here is when a card can help, not just which jobs appear on it.
play_notes=json.loads(Path("data/card_play_notes.json").read_text(encoding="utf-8"))
timing_labels = ["Instant / flash face","Sorcery face","Permanent setup","Land play","Activated ability",
                 "Triggered ability","Loyalty ability","Ability from hand","Restricted timing","Entry trigger","Death / leave trigger","Attack trigger","Combat damage trigger","Cast trigger","Upkeep / end-step trigger","Sacrifice to activate"]
mana_labels = ["Repeatable mana rock","Mana creature","One-shot mana","Land search to hand",
               "Land search to field","Fetch land","Grant flash to others"]
ready_labels = ["Interaction-capable","Instant / flash interaction","Permanent-based interaction","Hand ability interaction"]
def audit_card(card, item, roles):
    text = re.sub(r"\([^()]*\)","",card["oracle_text"])
    lower = text.lower()
    types = set(item["types"].split("; "))
    keywords = {k.lower() for k in card["keywords"]}
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    instant = "Instant / flash spell" in roles["deployment"]
    permanent = "Land" not in types and bool(types & {"Creature","Artifact","Planeswalker","Enchantment","Battle"})
    targetable = bool(roles["targets"] or roles["methods"])
    ability_text = re.sub(r'["“][^"”]+["”]', "", text)
    abilities = [line.lower().split(":",1) for line in ability_text.splitlines() if ":" in line]
    flags = [instant,"Sorcery" in card["all_types"],permanent,"Land" in types,
             bool(abilities),bool(re.search(r"\b(when|whenever|at the beginning of|at the end of)\b",lower)),
             "Planeswalker" in types and ":" in lower,"Hand ability" in roles["deployment"],
             bool(re.search(r"only (?:as a sorcery|during|once each turn|on your turn)",lower))]
    timing = [label for label,flag in zip(timing_labels, flags) if flag]
    mana = []
    trigger_lines = [line.lower() for line in lines if re.search(r"\b(?:when|whenever|at the beginning of)\b",line,re.I)]
    for label,pattern in [("Entry trigger",r"\benters?\b"),("Death / leave trigger",r"\b(?:dies?|leaves the battlefield)\b"),
                          ("Attack trigger",r"\battacks?\b"),("Combat damage trigger",r"deals? combat damage"),
                          ("Cast trigger",r"\bcasts?\b"),("Upkeep / end-step trigger",r"\b(?:upkeep|end step)\b")]:
        if any(re.search(pattern,clause.split(",",1)[0]) for clause in trigger_lines):timing.append(label)
    if any("sacrifice" in cost for cost,effect in abilities):timing.append("Sacrifice to activate")
    mana_abilities = [(cost,effect) for cost,effect in abilities if re.match(r"\s*add\b",effect)]
    def sacrifices_self(cost):
        return "sacrifice this" in cost or ("sacrifice "+card["name"].lower()) in cost
    if mana_abilities and "Land" not in types:
        if any(not sacrifices_self(cost) for cost,effect in mana_abilities):
            if "Creature" in types:mana.append("Mana creature")
            elif "Artifact" in types:mana.append("Repeatable mana rock")
        if any(sacrifices_self(cost) for cost,effect in mana_abilities):mana.append("One-shot mana")
    if not permanent and "Land" not in types and re.search(r"(?:^|\n|\. )add (?:\{|[a-z]+ mana)",lower):
        mana.append("One-shot mana")
    land_search = re.search(r"search your library for [^.]{0,100}\b(?:land|plains|island|swamp|mountain|forest)\b[^.]{0,100}",lower)
    cycling = any(k.endswith("cycling") and k not in {"cycling","typecycling","wizardcycling","slivercycling"} and
                 any(t in k for t in ["land","plains","island","swamp","mountain","forest"]) for k in keywords)
    if cycling or (land_search and re.search(r"(?:put|reveal)[^.]{0,130}(?:your hand|into your hand)",lower)):
        mana.append("Land search to hand")
    if land_search and re.search(r"put [^.]{0,150}(?:onto|to) the battlefield",lower):
        mana.append("Land search to field")
        if "Land" in types and any(sacrifices_self(cost) for cost,effect in abilities):mana.append("Fetch land")
    if re.search(r"(?:spells|cards) [^.]{0,100}(?:had flash|have flash)",lower):mana.append("Grant flash to others")
    ready = [label for label,flag in zip(ready_labels,[targetable,targetable and instant,targetable and permanent,
                targetable and "Hand ability" in roles["deployment"]]) if flag]
    # Costs and trigger clauses are quoted evidence. They are not a ruling on every mode.
    gates=[]
    for line in lines:
        if ":" in re.sub(r'["“][^"”]+["”]', "", line):
            before=line.split(":",1)[0]
            if len(before)<155:gates.append(before+":")
        trigger=re.match(r"(?:When(?:ever)?|At the beginning of|If|As long as|During)\b[^,\n]{2,140},",line,re.I)
        if trigger:gates.append(trigger[0].rstrip(","))
        restricted=re.search(r"(?:activate|cast|trigger)[^.]{0,45}\bonly\b[^.]{1,90}",line,re.I)
        if restricted:gates.append(restricted[0])
    note=play_notes.get(card["name"].split(" // ")[0],"")
    if note:gates.insert(0,note)
    faces = "; ".join(f["name"]+": "+(f["cost"] or "no printed cost") for f in card["faces"])
    result = dict(timing=timing, mana_access=mana, readiness=ready, conditions="; ".join(dict.fromkeys(gates)),
                  face_costs=faces, mv=item["mv"], types=card["all_types"], url=card["url"])
    return result


## Color and creature composition

A commander can permit colors its deck barely uses. Compare its identity to actual nonland slots, then compare colored casting symbols. Hybrid symbols split their weight; a colorless cost supplies no colored symbols. Creature types are printed types, not types gained from changeling or tokens.

In [ ]:
color_names = dict(zip("WUBRG",["White","Blue","Black","Red","Green"]))
color_buckets = [*color_names.values(),"Multicolor","Colorless"]
creature_buckets = ["Creature only","Artifact creature","Enchantment creature","Other multitype creature"]
def colored_symbols(cost):
    # A hybrid symbol is one symbol equivalent divided between its colored options.
    equivalents = Counter()
    for symbol in re.findall(r"\{([^}]+)\}",cost):
        options = [part for part in symbol.split("/") if part in color_names]
        for code in options:
            equivalents[color_names[code]] += 1 / len(options)
    return equivalents

def adjusted_subtypes(item):
    printed = set(filter(None,item["subtypes"].split("; ")))
    if "Human" in printed and len(printed)>1: printed.remove("Human")
    return printed

adjusted_leaders = set()
for profile in profiles:
    subtype_counts = Counter()
    for item in profile["audit"]:
        subtype_counts.update({subtype:item["quantity"] for subtype in adjusted_subtypes(item)})
    if subtype_counts:
        peak=max(subtype_counts.values())
        adjusted_leaders.update(k for k,n in subtype_counts.items() if n==peak)


In [ ]:
from collections import defaultdict
sections = {"Printed curve": curve_labels, **{v:labels for v,(_,labels) in role_axes.items()},
            "Card types":type_labels}
top_subtypes = sorted({t for p in profiles for t,qty in p["tribes"].items() if qty == max(p["tribes"].values())})
if top_subtypes: sections["Creature types"] = top_subtypes
sections.update({"Mana and access":mana_labels,"Timing and setup":timing_labels,"Interaction readiness":ready_labels})
sections.update({"Commander identity":list(color_names.values()),"Printed card colors":color_buckets,
                 "Casting-cost emphasis":list(color_names.values()),"Creature composition":creature_buckets})
if adjusted_leaders:sections["Human-adjusted tribes"]=sorted(adjusted_leaders)
strategy_cells, evidence_rows = [], []
for p,g in zip(profiles,groups):
    members = defaultdict(dict)
    pips = Counter()
    for item in p["audit"]:
        card = card_facts[normalize(item["name"])]
        roles,source = printed_roles(card)
        hits = [(view,label) for view,(key,_) in role_axes.items() for label in roles[key]]
        if item["curve_bin"]: hits.append(("Printed curve",item["curve_bin"]))
        hits.extend(("Card types",t) for t in item["types"].split("; ") if t)
        hits.extend(("Creature types",t) for t in item["subtypes"].split("; ") if t in top_subtypes)
        types=set(item["types"].split("; "))
        per_card=colored_symbols(card["printed_cost"])
        if item["zone"]=="Commander":
            hits.extend(("Commander identity",color_names[k]) for k in card["identity"])
        if "Land" not in types:
            colors=card["printed_colors"]
            bucket="Multicolor" if len(colors)>1 else color_names[colors[0]] if colors else "Colorless"
            hits.append(("Printed card colors",bucket))
            pips.update({k:qty*item["quantity"] for k,qty in per_card.items()})
            hits.extend(("Casting-cost emphasis",k) for k in per_card)
        if "Creature" in types:
            others=types-{"Creature"}
            creature_mix=("Creature only" if not others else "Artifact creature" if others=={"Artifact"} else
                          "Enchantment creature" if others=={"Enchantment"} else "Other multitype creature")
            hits.append(("Creature composition",creature_mix))
            hits.extend(("Human-adjusted tribes",t) for t in adjusted_subtypes(item) if t in adjusted_leaders)
        audit = audit_card(card,item,roles)
        for view,field in [("Mana and access","mana_access"),("Timing and setup","timing"),("Interaction readiness","readiness")]:
            hits.extend((view,value) for value in audit[field])
        for view,role in hits:
            members[(view,role)][card["oracle_id"]] = (item["name"], item["quantity"])
        evidence_rows.append(dict(deck=p["name"],name=item["name"],zone=item["zone"],quantity=item["quantity"],
                   group=g, **{k:"; ".join(v) if isinstance(v,list) else v for k,v in audit.items()},
                   printed_colors="; ".join(color_names[k] for k in card["printed_colors"]) or "Colorless",
                   color_identity="; ".join(color_names[k] for k in card["identity"]) or "Colorless", printed_cost=card["printed_cost"],
                   source=source,oracle_text=card["oracle_text"],oracle_id=card["oracle_id"],
                   **{key:"; ".join(sorted(vals)) for key,vals in roles.items()}))
    for view,labels in sections.items():
        for order,label in enumerate(labels):
            found=sorted(members[(view,label)].values())
            count=sum(q for n,q in found) if view in ("Printed curve","Card types","Creature types","Printed card colors","Creature composition","Human-adjusted tribes") else len(found)
            if view=="Commander identity": count=int(bool(found))
            if view=="Casting-cost emphasis": count=round(100*pips[label]/sum(pips.values())) if pips else None
            strategy_cells.append(dict(view=view,deck=p["name"],group=g,category=label,position=order,
                                   count=count, slot_share=(count/(100-p["types"]["Land"]) if view=="Printed card colors" and p["types"]["Land"]<100 else ""),
                                   symbol_equivalents=pips[label] if view=="Casting-cost emphasis" else "",
                                   colored_symbol_total=sum(pips.values()) if view=="Casting-cost emphasis" else "",
                                   cards="; ".join(n for n,q in found)))
group_rank = {"Your deck":0,"Bracket 4":1,"Bracket 3":2,"Bracket 2":3,"Stock precon":4}
deck_order = {p["name"]:i for i,p in enumerate(sorted(profiles,key=lambda p:(
    group_rank["Your deck" if p["group"]=="User" else ("Bracket "+p["bracket"] if p["group"]=="Player-built" else "Stock precon")],p["name"])))}
for row in strategy_cells:
    row["deck_order"] = deck_order[row["deck"]]
    row["group_order"] = group_rank[row["group"]]
    row["field_label"] = f'{row["deck_order"]+1:02} | {row["group"]} | {row["deck"]}'
strategy_cells.sort(key=lambda r:(r["deck_order"], r["position"]))
write_csv(out_dir/"strategy_cells.csv",strategy_cells)
write_csv(out_dir/"strategy_card_evidence.csv",evidence_rows)

# Matched pairs hold the player list fixed and vary only the public opponent for comparison.
user_cells = {(r["view"],r["category"]):r for r in strategy_cells if r["group"]=="Your deck"}
pair_rows = []
for reference in strategy_cells:
    if reference["group"]!="Your deck":
        own = user_cells[(reference["view"],reference["category"])]
        for side, row in [("Your deck",own),("Reference",reference)]:
            pair_rows.append(dict(view=row["view"], reference=reference["deck"], reference_choice=reference["field_label"], cohort=reference["group"],
                                  side=side, deck=row["deck"], category=row["category"], position=row["position"],
                                  count=row["count"], cards=row["cards"]))
write_csv(out_dir/"strategy_pairs.csv",pair_rows)


## How does the deck deploy, respond, and rebuild?

Separate the library plan from command-zone support. Reuse can require a surviving permanent, while recursion can recover one that was lost. The player column is an actual deck; bracket columns are arithmetic means, not power scores.

In [ ]:
# A single review per list; authored interpretations are only used for an exact public snapshot.
def supporting_names(rows, key, role, limit=3):
    matches = sorted(r["name"] for r in rows if role in r[key].split("; "))
    return ", ".join(matches[:limit])

def review_list(profile, group, rows, note):
    library = [r for r in rows if r["zone"]=="Deck"]
    commander = [r for r in rows if r["zone"]=="Commander"]
    count_role = lambda key,tag:len({r["oracle_id"] for r in rows if tag in r[key].split("; ")})
    ready = count_role("readiness","Interaction-capable")
    fast = count_role("readiness","Instant / flash interaction")
    def route(key, tags):
        portions=[]
        for role in tags:
            names=supporting_names(library,key,role,2)
            if names:portions.append(f"{role}: {names}.")
        return " ".join(portions) or "No library card tagged for these routes."
    alternate=route("deployment",["Deploy other cards","Own alternate payment","Spell discounts","Ability discounts","Hand ability"])
    if not note:
        named_routes=[r for r in library if "Deploy other cards" in r["deployment"] and r["name"] in play_notes]
        if named_routes:
            example=sorted(named_routes,key=lambda r:r["name"])[0]["name"]
            alternate += f" {example}: {play_notes[example]}"
    resources=route("recovery",["Recurring draw","Entry draw","Untap support","Copy / imprint","Token production"])
    rebuild=route("recovery",["Recover other cards","Own-card reuse","Blink / return","Return own to hand"])
    coverage = [r for r in commander if r["recovery"] or r["deployment"]]
    checks=[]
    for r in coverage:
        for key,tags in [("recovery",r["recovery"]),("deployment",r["deployment"])]:
            for tag in tags.split("; "):
                if tag and tag not in ("Instant / flash spell","Own alternate payment"):
                    others=[c for c in library if tag in c[key].split("; ")]
                    checks.append((len(others),r["name"],tag,", ".join(c["name"] for c in others[:2])))
    dependence = profile["adds"]
    interruption = "No specific failure point inferred from the printed tags. Inspect what has to resolve or survive before counting an effect as available."
    if checks:
        number,name,tag,names=min(checks)
        dependence += f" Overlap with the library: {tag} ({number} cards)."
        pressure={"Recover other cards":"Graveyard exile can deny recovery targets before they return.","Token production":"A board wipe can erase the tokens without removing a surviving token engine.","Recurring draw":"Removing the draw source before its next trigger reduces the hand refill.","Deploy other cards":"Removing the enabler can leave the payload in hand or in the graveyard.","Copy / imprint":"Removing the imprinted or copied resource can interrupt the reuse route."}
        interruption=pressure.get(tag,"Remove the commander before its next activation or trigger; previously triggered abilities can survive removal.")+" "+(f"Removing {name} alone leaves {names} in the library as other sources of this capability." if names else "No library card was tagged for the same route.")
    if note:
        assert set(note["anchors"])<={r["name"] for r in rows},(profile["name"],note["anchors"])
        alternate=note["deployment_line"]+" "+alternate
        rebuild=note["resource"]+" "+rebuild
        interruption=note["failure"]
    summary = dict(deck=profile["name"],group=group,game_plan=profile["without"],alternate_routes=alternate,
                   commander_dependence=dependence,resource_reuse=resources,recovery=rebuild,interruption=interruption,
                   library_nonlands=profile["spells"],mean_mv=round(profile["mean_mv"],3),lands=profile["types"]["Land"],
                   mana_rocks=count_role("mana_access","Repeatable mana rock"),interaction=ready,instant_interaction=fast)
    return summary
review_rows=[]
for p,g,sections_by_zone in zip(profiles,groups,deck_sections):
    signature_parts=sorted(f"{q} {n}" for n,q in sections_by_zone["Commander"].items())+["Deck"]+sorted(f"{q} {n}" for n,q in sections_by_zone["Deck"].items())
    signature=hashlib.sha256("\n".join(signature_parts).encode()).hexdigest()
    review_rows.append(review_list(p,g,[r for r in evidence_rows if r["deck"]==p["name"]],
                                  reviews.get(signature) if g!="Your deck" else None))
write_csv(out_dir/"deck_routes.csv",review_rows)

summary_metrics = {"mean_mv":"Mean library MV","library_nonlands":"Nonland library slots","lands":"Front-face lands",
                   "mana_rocks":"Repeatable mana rocks","interaction":"Interaction-capable cards","instant_interaction":"Instant / flash interaction"}
group_rows=[]
for group in group_rank:
    members=[r for r in review_rows if r["group"]==group]
    if not members:continue
    for key,label in summary_metrics.items():
        group_rows.append(dict(group=group,metric=label,value=round(mean(r[key] for r in members),2),lists=len(members),group_order=group_rank[group]))
    for v,k in [("Printed curve","0-1"),("Printed curve","6+"),("Deployment routes","Deploy other cards"),("Deployment routes","Spell discounts"),("Interaction targets","Stack abilities"),("Interaction methods","Multi-permanent reset"),("Interaction methods","Taxes / restrictions"),("Recovery routes","Recover other cards"),("Recovery routes","Blink / return"),("Recovery routes","Entry draw")]:
        values=[r["count"] for r in strategy_cells if r["group"]==group and r["view"]==v and r["category"]==k]
        group_rows.append(dict(group=group,metric=("MV "+k if v=="Printed curve" else k),value=round(mean(values),2),lists=len(members),group_order=group_rank[group]))
write_csv(out_dir/"cohort_measures.csv",group_rows)


In [ ]:
# Detail sheets for the native dashboards inherit the same counts, reviews and card audit as the browser.
detail_rows=[]
def add_detail(section, deck, heading, detail="", cost="", timing="", conditions="", url="", group=""):
    deck_index = deck_order.get(deck, group_rank.get(group,0))
    deck_label = (f"{deck_index+1:02} | {deck}" if deck in deck_order else (f"{group_rank[group]+1:02} | {group}" if group else deck))
    detail_rows.append(dict(section=section, deck=deck, deck_label=deck_label, deck_order=deck_index,
                            heading=heading, detail=detail, cost=cost, timing=timing, conditions=conditions, url=url))
for review in review_rows:
    for heading,key in [("Library plan","game_plan"),("Alternate routes","alternate_routes"),("Commander dependence","commander_dependence"),
                        ("Resource reuse","resource_reuse"),("Recovery","recovery"),("What interrupts it","interruption")]:
        add_detail("Deck routes",review["deck"],heading,review[key])
for r in evidence_rows:
    tags="; ".join(r[k] for k in ["deployment","targets","methods","recovery","mana_access","readiness"] if r[k])
    add_detail("Card evidence",r["deck"],r["name"]+(" (commander)" if r["zone"]=="Commander" else ""),
               tags+"\n"+r["oracle_text"],r["face_costs"],r["timing"],r["conditions"],r["url"])
for r in group_rows:
    add_detail("Bracket means",r["group"],r["metric"],str(r["value"]),conditions=f'{r["lists"]} list(s)',group=r["group"])
for k,v in json.loads(Path("data/capability_definitions.json").read_text(encoding="utf-8")).items():
    add_detail("Definitions","All decks",k,v)
write_csv(out_dir/"review_details.csv",detail_rows)
